# APHELION HYDRA — SUPER INSANE Training on Google Colab

**Full pipeline: upload → install → configure → train → download checkpoint**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have your `Aphelion` folder zipped and ready to upload

---

## STEP 1 — Verify GPU (must say Tesla T4)

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("NO GPU! Go to Runtime → Change runtime type → T4 GPU")

## STEP 2 — Upload your project

**On your Windows PC, zip the Aphelion folder:**
```
Right-click C:\Users\marti\Aphelion → Send to → Compressed (zipped) folder
```
Or in PowerShell:
```powershell
cd C:\Users\marti
Compress-Archive -Path .\Aphelion\aphelion, .\Aphelion\scripts, .\Aphelion\data, .\Aphelion\pyproject.toml, .\Aphelion\requirements.txt -DestinationPath .\Aphelion_upload.zip -Force
```

Then run the cell below and click **Choose Files** to upload `Aphelion_upload.zip`

In [ ]:
from google.colab import files
import os, zipfile, shutil

# Clean previous uploads
if os.path.exists('/content/Aphelion'):
    shutil.rmtree('/content/Aphelion')

print("Upload your Aphelion_upload.zip (or Aphelion.zip):")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\nExtracting {zip_name}...")

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Handle nested folder structure
if os.path.exists('/content/Aphelion'):
    PROJECT = '/content/Aphelion'
elif os.path.exists('/content/aphelion'):
    PROJECT = '/content/aphelion'
else:
    # Zip might have files at root
    PROJECT = '/content'

os.chdir(PROJECT)
print(f"\nProject root: {PROJECT}")
print(f"Contents: {os.listdir('.')}")

### Alternative: Upload from Google Drive (faster for big files)

In [ ]:
# UNCOMMENT these lines if you prefer Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/Aphelion_upload.zip /content/
# !cd /content && unzip -o Aphelion_upload.zip
# %cd /content/Aphelion

## STEP 3 — Install dependencies

In [ ]:
%%time
# Install the project in editable mode with ML dependencies
!pip install -e ".[ml]" -q
!pip install loguru -q

# Verify imports
from aphelion.intelligence.hydra.ensemble import HydraGate, EnsembleConfig
from aphelion.intelligence.hydra.trainer import HydraTrainer, TrainerConfig
print("All imports OK")

## STEP 4 — Verify data is present

In [ ]:
import os
import pandas as pd

data_dir = 'data/bars'
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"{data_dir} not found! Make sure data/bars/ is in the zip.")

csv_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.csv')])
print(f"Found {len(csv_files)} data files:\n")

total_bars = 0
total_mb = 0
for f in csv_files:
    path = os.path.join(data_dir, f)
    size_mb = os.path.getsize(path) / 1024 / 1024
    rows = sum(1 for _ in open(path)) - 1
    total_bars += rows
    total_mb += size_mb
    print(f"  {f:35s} {rows:>10,} bars  ({size_mb:.1f} MB)")

print(f"\nTotal: {total_bars:,} bars, {total_mb:.1f} MB")

# Preview the main training file
main_csv = os.path.join(data_dir, 'xauusd_m5.csv')
if os.path.exists(main_csv):
    df = pd.read_csv(main_csv)
    print(f"\nMain training data (xauusd_m5.csv): {len(df):,} bars")
    print(f"Date range: {df.iloc[0]['timestamp']} → {df.iloc[-1]['timestamp']}")
    print(f"Price range: ${df['close'].min():.2f} — ${df['close'].max():.2f}")
    df.head(3)

## STEP 5 — Quick smoke test (CPU, 2 epochs, tiny model)

This takes ~30 seconds and verifies everything works before we commit GPU time.

In [ ]:
%%time
!python scripts/train_hydra.py --epochs 2 --batch-size 16
print("\n Smoke test passed!")

## STEP 6 — REAL TRAINING (SUPER INSANE)

This trains the **full 6-model ensemble** on real XAUUSD M5 data with:
- Label smoothing focal loss
- Mixup augmentation
- Gradient accumulation (4x effective batch)
- All 6 auxiliary sub-model losses
- MoE load balancing
- Diversity loss
- Linear warmup + cosine annealing
- SWA in final phase

**Expected time: ~2-4 hours on T4 for 100 epochs with 100K bars**

In [ ]:
# ============================================================
#  TRAINING CONFIGURATION — EDIT THESE IF NEEDED
# ============================================================

DATA_FILE   = "data/bars/xauusd_m5.csv"   # Main training data
EPOCHS      = 100                           # Start with 100, increase if still improving
BATCH_SIZE  = 64                            # 64 works well on T4 (16GB VRAM)
CHECKPOINT  = "models/hydra"                # Where to save checkpoints

In [ ]:
%%time
import torch
torch.cuda.empty_cache()

cmd = (
    f"python scripts/train_hydra.py"
    f" --data {DATA_FILE}"
    f" --full"
    f" --epochs {EPOCHS}"
    f" --batch-size {BATCH_SIZE}"
    f" --checkpoint-dir {CHECKPOINT}"
)

print(f"Running: {cmd}\n")
!{cmd}

## STEP 7 — Check training results

In [ ]:
import torch
import os

ckpt_dir = 'models/hydra'
files = sorted(os.listdir(ckpt_dir)) if os.path.exists(ckpt_dir) else []
print("Checkpoints saved:")
for f in files:
    size = os.path.getsize(os.path.join(ckpt_dir, f)) / 1024 / 1024
    print(f"  {f:45s} ({size:.1f} MB)")

# Load best checkpoint
best_path = os.path.join(ckpt_dir, 'hydra_ensemble_best_sharpe.pt')
if not os.path.exists(best_path):
    best_path = os.path.join(ckpt_dir, 'hydra_ensemble_best_loss.pt')

if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    print(f"\n{'='*60}")
    print(f"BEST CHECKPOINT: {os.path.basename(best_path)}")
    print(f"{'='*60}")
    print(f"  Epoch:          {ckpt.get('epoch', '?')}")
    print(f"  Best val Sharpe: {ckpt.get('best_val_sharpe', '?'):.4f}")
    print(f"  Best val loss:   {ckpt.get('best_val_loss', '?'):.4f}")
    
    # Plot training curves
    train_hist = ckpt.get('train_history', [])
    val_hist = ckpt.get('val_history', [])
    if train_hist and val_hist:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        epochs = range(1, len(train_hist) + 1)
        
        # Loss
        axes[0].plot(epochs, [h['loss'] for h in train_hist], label='Train')
        axes[0].plot(epochs, [h['loss'] for h in val_hist], label='Val')
        axes[0].set_title('Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Accuracy
        axes[1].plot(epochs, [h['accuracy']*100 for h in train_hist], label='Train')
        axes[1].plot(epochs, [h['accuracy']*100 for h in val_hist], label='Val')
        axes[1].set_title('Accuracy (%)')
        axes[1].set_xlabel('Epoch')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Sharpe
        axes[2].plot(epochs, [h.get('sharpe_proxy', 0) for h in val_hist], color='green')
        axes[2].set_title('Validation Sharpe Proxy')
        axes[2].set_xlabel('Epoch')
        axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No checkpoint found!")

## STEP 8 — Download trained model to your PC

In [ ]:
import shutil, os
from google.colab import files

# Zip all checkpoints
ckpt_dir = 'models/hydra'
zip_name = 'hydra_checkpoints'

if os.path.exists(ckpt_dir):
    shutil.make_archive(zip_name, 'zip', ckpt_dir)
    size_mb = os.path.getsize(f'{zip_name}.zip') / 1024 / 1024
    print(f"Downloading hydra_checkpoints.zip ({size_mb:.1f} MB)...")
    files.download(f'{zip_name}.zip')
else:
    print("No checkpoints to download!")

### Alternative: Save to Google Drive

In [ ]:
# UNCOMMENT to save checkpoints to Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r models/hydra /content/drive/MyDrive/hydra_checkpoints/
# print("Saved to Google Drive: MyDrive/hydra_checkpoints/")

## STEP 9 — (Optional) Train on MORE data

Once the M5 model works, train on M1 data for maximum resolution.
Warning: 100K M1 bars = much larger dataset, may need 2+ hours.

In [ ]:
# # M1 training (uncomment to run)
# !python scripts/train_hydra.py \
#     --data data/bars/xauusd_m1.csv \
#     --full \
#     --epochs 150 \
#     --batch-size 64 \
#     --checkpoint-dir models/hydra_m1

---

## After downloading the checkpoint:

1. Unzip `hydra_checkpoints.zip` into `C:\Users\marti\Aphelion\models\hydra\`
2. The system will auto-load the best checkpoint for live/paper trading
3. Key files:
   - `hydra_ensemble_best_sharpe.pt` — Best risk-adjusted model
   - `hydra_ensemble_best_loss.pt` — Best raw accuracy model
   - `hydra_ensemble_latest.pt` — Last epoch checkpoint
   - `hydra_ensemble_swa_final.pt` — SWA averaged model (if training ran long enough)